In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import re
from pathlib import Path

DATA_DIR = Path('final_results_copy')


In [ ]:
# ── Algorithm name normalisation ──────────────────────────────────────────────

SEQ_PATTERN = re.compile(
    r'\s*\((water only|water|fat fraction|fat_fraction|dixon|Dixon|both channels?|both)\)\s*$',
    re.IGNORECASE
)

def base_name(name: str) -> str:
    return SEQ_PATTERN.sub('', name).strip()

def extract_seq(name):
    m = re.search(r'\((water only|water|fat fraction|dixon|both[^)]*?)\)', name, re.IGNORECASE)
    if m:
        s = m.group(1).strip().lower()
        return {'water': 'Water', 'water only': 'Water',
                'fat fraction': 'Fat fraction',
                'dixon': 'Dixon'}.get(s, s.capitalize())
    return 'Single'

# Rename raw CSV names to display names
RENAME = {'Hirriririir': 'Multimodal-multiethnic'}

def display_name(name: str) -> str:
    return RENAME.get(base_name(name), base_name(name))

# Algorithm display order (top = best overall)
ALG_ORDER = [
    'MuscleMap WB',
    'MuscleMap Thigh',
    'MM WB + MedSAM bbox',
    'MM WB + SLM-SAM2',
    'MuSeg',
    'Multimodal-multiethnic',
    'MedCLIP-SAMv2 Text+Boxes',
    'MM WB + MedSAM mask',
    'MedCLIP-SAMv2',
    'Dafne + MedSAM',
    'Dafne',
    'MedSegDiff',
]

# Algorithm family -> label  (aligned with Table 1)
FAMILY = {
    'MuscleMap WB':             'U-Net',
    'MuscleMap Thigh':          'U-Net',
    'MuSeg':                    'nnU-Net',
    'Multimodal-multiethnic':   'SegResNet (MONAI)',
    'MM WB + MedSAM bbox':      'SAM-based',
    'MM WB + MedSAM mask':      'SAM-based',
    'MM WB + SLM-SAM2':         'SAM-based',
    'MedCLIP-SAMv2':            'SAM-based',
    'MedCLIP-SAMv2 Text+Boxes': 'SAM-based',
    'Dafne + MedSAM':           'Federated DL',
    'Dafne':                    'Federated DL',
    'MedSegDiff':               'Diffusion',
}

FAMILY_COLORS = {
    'U-Net':             '#1f77b4',
    'nnU-Net':           '#aec7e8',
    'SegResNet (MONAI)': '#9467bd',
    'SAM-based':         '#ff7f0e',
    'Federated DL':      '#2ca02c',
    'Diffusion':         '#d62728',
}

DATASETS = {
    'MyoSegmenTUM': 'overall_means_myosegmentum.csv',
    'Pathological': 'overall_means_P_only.csv',
    'AIPS':         'overall_means_asian.csv',
    'Sheffield':    'overall_means_sheffield.csv',
    'Augmented':    'overall_means_augmented.csv',
}

SEQ_ORDER   = ['Water', 'Fat fraction', 'Dixon', 'Single']
SEQ_HATCHES = {'Water': '', 'Fat fraction': '///', 'Dixon': 'xxx', 'Single': ''}


In [ ]:
# ═══════════════════════════════════════════════════════════════════════════════
# Bar chart per dataset: Dice by algorithm, grouped by sequence
# ═══════════════════════════════════════════════════════════════════════════════

for ds_name, fname in DATASETS.items():
    df_raw = pd.read_csv(DATA_DIR / fname)
    df_raw['base']     = df_raw['algorithm'].apply(display_name)
    df_raw['sequence'] = df_raw['algorithm'].apply(extract_seq)

    df_bar = df_raw[df_raw['base'].isin(ALG_ORDER)].copy()
    if df_bar.empty:
        print(f'No matching algorithms for {ds_name}, skipping.')
        continue

    # Sort algorithms by mean Dice on this dataset (descending)
    alg_mean      = df_bar.groupby('base')['dice'].mean()
    alg_order_bar = alg_mean.sort_values(ascending=False).index.tolist()
    seq_present   = [s for s in SEQ_ORDER if s in df_bar['sequence'].unique()]
    n_seq         = len(seq_present)
    x             = np.arange(len(alg_order_bar))
    width         = 0.8 / n_seq

    fig, ax = plt.subplots(figsize=(max(10, len(alg_order_bar) * 1.2), 7))

    for k, seq in enumerate(seq_present):
        subset = df_bar[df_bar['sequence'] == seq].set_index('base')['dice']
        vals   = [subset.get(a, np.nan) for a in alg_order_bar]
        colors = [FAMILY_COLORS[FAMILY.get(a, 'U-Net')] for a in alg_order_bar]
        offset = (k - n_seq / 2 + 0.5) * width
        ax.bar(x + offset, vals, width * 0.92, label=seq,
               color=colors, hatch=SEQ_HATCHES[seq],
               edgecolor='white', linewidth=0.5, alpha=0.85)

    ax.set_xticks(x)
    ax.set_xticklabels(alg_order_bar, rotation=35, ha='right', fontsize=14)
    ax.set_ylabel('Dice Score', fontsize=16)
    ax.set_ylim(0, 1.0)
    ax.set_title(f'{ds_name} \u2014 Dice Score by Algorithm and MRI Sequence',
                 fontsize=16, fontweight='bold')
    ax.axhline(0, color='black', linewidth=0.5)
    ax.yaxis.grid(True, linestyle='--', alpha=0.4)
    ax.set_axisbelow(True)

    # Sequence legend (upper right, inside plot) -- only when >1 sequence present
    if n_seq > 1:
        seq_handles = [mpatches.Patch(facecolor='grey', hatch=SEQ_HATCHES[s],
                                      label=s, alpha=0.85)
                       for s in seq_present]
        ax.legend(handles=seq_handles, title='Sequence',
                  loc='upper right', framealpha=0.9, fontsize=13,
                  title_fontsize=13)

    plt.tight_layout()
    slug = ds_name.lower().replace(' ', '_')
    plt.savefig(f'bar_{slug}_by_sequence.pdf', bbox_inches='tight')
    plt.savefig(f'bar_{slug}_by_sequence.png', dpi=150, bbox_inches='tight')
    plt.savefig(f'bar_{slug}_by_sequence.tif', dpi=300, bbox_inches='tight',
                format='tiff', pil_kwargs={'compression': 'tiff_lzw'})
    plt.show()
    print(f'Saved bar_{slug}_by_sequence  .pdf / .png / .tif')

    # Algorithm family legend as a separate figure
    families_present = {FAMILY.get(a) for a in alg_order_bar} - {None}
    fam_handles = [mpatches.Patch(facecolor=FAMILY_COLORS[f], label=f)
                   for f in FAMILY_COLORS if f in families_present]
    fig_leg, ax_leg = plt.subplots(figsize=(6, 1 + len(fam_handles) * 0.4))
    ax_leg.axis('off')
    ax_leg.legend(handles=fam_handles, title='Algorithm family',
                  loc='center', ncol=min(len(fam_handles), 3),
                  fontsize=13, title_fontsize=13, framealpha=0.9)
    plt.tight_layout()
    plt.savefig(f'legend_{slug}_family.pdf', bbox_inches='tight')
    plt.savefig(f'legend_{slug}_family.png', dpi=150, bbox_inches='tight')
    plt.savefig(f'legend_{slug}_family.tif', dpi=300, bbox_inches='tight',
                format='tiff', pil_kwargs={'compression': 'tiff_lzw'})
    plt.show()
    print(f'Saved legend_{slug}_family.pdf / .png')
